In [ ]:
!pip -q install "transformers==5.0.0" "accelerate>=1.10.1" 2>&1 | tail -3


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__, "| cpus", os.cpu_count())
print(subprocess.run(["free", "-m"], capture_output=True, text=True).stdout)

found = sorted(Path("/kaggle/input").rglob("repo/scripts/kernel_gates.py"))
assert found, "the dataset has no kernel_gates.py"
sys.path.insert(0, str(found[0].parent))
from kernel_gates import run_bounded  # noqa: E402

os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"  # as in both T4 runs
os.environ["TOKENIZERS_PARALLELISM"] = "false"
PROBE = Path("/kaggle/working/codec_probe.py")
PROBE.write_text("""
import faulthandler, resource, sys, time
faulthandler.dump_traceback_later(60, repeat=True, file=sys.stderr)
started = time.time()
from transformers import AutoModel
codec = AutoModel.from_pretrained(sys.argv[1], trust_remote_code=True, codec_weight_dtype="bf16")
faulthandler.cancel_dump_traceback_later()
params = sum(p.numel() for p in codec.parameters())
print(f"LOADED in {time.time() - started:.0f}s, {params / 1e9:.2f}B parameters, "
      f"peak RSS {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:.1f} GiB", flush=True)
""")
RESULTS = {}


In [ ]:
from huggingface_hub import snapshot_download

start = time.time()
CODEC_DIR = Path(snapshot_download(repo_id="OpenMOSS-Team/MOSS-Audio-Tokenizer-v2", revision="f6e20e543b33d2c252a7ef71bdf8aa71e5ff9169"))
print(f"codec in {(time.time() - start) / 60:.1f} min")


In [ ]:
code, seconds, timed_out = run_bounded([sys.executable, str(PROBE), str(CODEC_DIR)], timeout=1500,
                                       drain=True, monitor_every=60, tee=Path("/kaggle/working/drained.log"))
RESULTS["drained_cold_cache"] = {"exit": code, "seconds": round(seconds), "timed_out": timed_out}


In [ ]:
code, seconds, timed_out = run_bounded([sys.executable, str(PROBE), str(CODEC_DIR)], timeout=900,
                                       drain=False, monitor_every=60)
RESULTS["inherited_warm_cache"] = {"exit": code, "seconds": round(seconds), "timed_out": timed_out}


In [ ]:
print(json.dumps(RESULTS, indent=2))
drained, inherited = RESULTS["drained_cold_cache"], RESULTS["inherited_warm_cache"]
if not drained["timed_out"] and drained["exit"] == 0 and inherited["timed_out"]:
    reading = "drained finished, inherited hung: consistent with a writer blocked on an undrained pipe"
elif not drained["timed_out"] and not inherited["timed_out"]:
    reading = "both finished: not reproduced on a CPU box"
else:
    reading = "the drained load also failed: something other than output - see the wait channels above"
print("READING:", reading)
RESULTS["reading"] = reading
Path("/kaggle/working/probe.json").write_text(json.dumps(RESULTS, indent=2))
